# Notebook 6: Batch Train All Continual SD-LoRA Adapters

Bu notebook, Notebook 2 akisini baz alir ve 8 adaptoru tek kosuda sirayla egitir.

Akis:
1. Repo bootstrap ve notebook 2 helper'larini yukle.
2. Adapter matrisi ve sirayi tanimla.
3. Her adapter icin Notebook 2 parametre, egitim, OOD kalibrasyon, export ve final evaluation hucrelerini calistir.
4. Her run'i repo'ya kaydet ve summary yazdir.

Not: Bu defter auto-disconnect kapali calisir; loop tamamlanana kadar runtime acik kalir.

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

NOTEBOOK_NAME = "6_train_all_continual_sd_lora_adapters.ipynb"
NOTEBOOK_FILENAME = "6_train_all_continual_sd_lora_adapters.executed.ipynb"
AUTO_DISCONNECT_RUNTIME = False
AUTO_PUSH_TO_GITHUB = True
ENABLE_BAYESIAN_OPTIMIZATION = True
MANUAL_PARAM_OVERRIDES = {}

ADAPTER_RECS = {
    "grape__fruit": {"crop": "grape", "part": "fruit", "ood": "data/prepared_runtime_datasets/grape__fruit/ood", "oe": "data/prepared_runtime_datasets/grape__fruit/oe", "oe_enabled": True, "oe_w": 0.22, "allow_under_min": False, "overrides": {"EPOCHS": 34, "BATCH_SIZE": 48, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.16, "OOD_FACTOR": 3.5, "LABEL_SMOOTHING": 0.08}},
    "grape__leaf": {"crop": "grape", "part": "leaf", "ood": "data/prepared_runtime_datasets/grape__leaf/ood", "oe": "data/prepared_runtime_datasets/grape__leaf/oe", "oe_enabled": True, "oe_w": 0.22, "allow_under_min": False, "overrides": {"EPOCHS": 30, "BATCH_SIZE": 56, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.14, "OOD_FACTOR": 3.5, "LABEL_SMOOTHING": 0.08}},
    "strawberry__fruit": {"crop": "strawberry", "part": "fruit", "ood": "data/prepared_runtime_datasets/strawberry__fruit/ood", "oe": "data/prepared_runtime_datasets/strawberry__fruit/oe", "oe_enabled": True, "oe_w": 0.20, "allow_under_min": True, "overrides": {"EPOCHS": 38, "BATCH_SIZE": 40, "LEARNING_RATE": 8e-5, "LORA_R": 20, "LORA_ALPHA": 20, "LORA_DROPOUT": 0.20, "OOD_FACTOR": 4.0, "LABEL_SMOOTHING": 0.10}},
    "strawberry__leaf": {"crop": "strawberry", "part": "leaf", "ood": "data/prepared_runtime_datasets/strawberry__leaf/ood", "oe": "data/prepared_runtime_datasets/strawberry__leaf/oe", "oe_enabled": True, "oe_w": 0.25, "allow_under_min": False, "overrides": {"EPOCHS": 24, "BATCH_SIZE": 88, "LEARNING_RATE": 1.2e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.18, "OOD_FACTOR": 4.0, "LABEL_SMOOTHING": 0.10}},
    "tomato__fruit": {"crop": "tomato", "part": "fruit", "ood": "data/prepared_runtime_datasets/tomato__fruit/ood", "oe": "data/prepared_runtime_datasets/tomato__fruit/oe", "oe_enabled": True, "oe_w": 0.18, "allow_under_min": False, "overrides": {"EPOCHS": 32, "BATCH_SIZE": 64, "LEARNING_RATE": 9e-5, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.16, "OOD_FACTOR": 3.5, "LABEL_SMOOTHING": 0.08}},
    "tomato__leaf": {"crop": "tomato", "part": "leaf", "ood": "data/prepared_runtime_datasets/tomato__leaf/ood", "oe": "data/prepared_runtime_datasets/tomato__leaf/oe", "oe_enabled": True, "oe_w": 0.18, "allow_under_min": False, "overrides": {"EPOCHS": 24, "BATCH_SIZE": 88, "LEARNING_RATE": 1.1e-4, "LORA_R": 32, "LORA_ALPHA": 32, "LORA_DROPOUT": 0.12, "OOD_FACTOR": 3.5, "LABEL_SMOOTHING": 0.08}},
    "apricot__fruit": {"crop": "apricot", "part": "fruit", "ood": "data/ood_dataset/final/apricot__fruit_ood_final", "oe": "", "oe_enabled": False, "oe_w": 0.10, "allow_under_min": False, "overrides": {"EPOCHS": 36, "BATCH_SIZE": 40, "LEARNING_RATE": 1e-4, "LORA_R": 20, "LORA_ALPHA": 20, "LORA_DROPOUT": 0.25, "OOD_FACTOR": 4.5, "LABEL_SMOOTHING": 0.15}},
    "apricot__leaf": {"crop": "apricot", "part": "leaf", "ood": "data/ood_dataset/final/apricot__leaf_ood_final", "oe": "data/oe_dataset/apricot_leaf_oe_unsupported_leaf_candidates", "oe_enabled": True, "oe_w": 0.30, "allow_under_min": False, "overrides": {"EPOCHS": 38, "BATCH_SIZE": 52, "LEARNING_RATE": 1.2e-4, "LORA_R": 26, "LORA_ALPHA": 26, "LORA_DROPOUT": 0.22, "OOD_FACTOR": 5.5, "LABEL_SMOOTHING": 0.15}},
}

NB6_ADAPTER_SEQUENCE = [
    "apricot__leaf",
    "apricot__fruit",
    "strawberry__leaf",
    "strawberry__fruit",
    "grape__leaf",
    "grape__fruit",
    "tomato__leaf",
    "tomato__fruit",
]

run_cell_script('nb2_cell01_bootstrap_access.py', globals())

In [ ]:
import json

NB6_RESULTS = {}

for index, adapter_key in enumerate(NB6_ADAPTER_SEQUENCE, start=1):
    print(f"\n[NB6] Starting {index}/{len(NB6_ADAPTER_SEQUENCE)}: {adapter_key}")
    ADAPTER_KEY = adapter_key
    MANUAL_PARAM_OVERRIDES = {}
    try:
        run_cell_script('nb2_cell03_runtime_setup.py', globals())
        run_cell_script('nb2_cell04_parameter_resolution.py', globals())
        run_cell_script('nb2_cell05_access_check.py', globals())
        run_cell_script('nb2_cell06_dataset_validation.py', globals())
        run_cell_script('nb2_cell07_engine_init.py', globals())
        run_cell_script('nb2_cell08_ood_config_verify.py', globals())
        run_cell_script('nb2_cell09_training.py', globals())
        run_cell_script('nb2_cell10_ood_calibration.py', globals())
        run_cell_script('nb2_cell11_adapter_save.py', globals())
        run_cell_script('nb2_cell12_final_evaluation.py', globals())
        NB6_RESULTS[adapter_key] = {
            "status": "ok",
            "run_id": str(globals().get("RUN_ID", "")),
            "crop": str(globals().get("CROP_NAME", "")),
            "part": str(globals().get("PART_NAME", "")),
        }
    except Exception as exc:
        NB6_RESULTS[adapter_key] = {
            "status": "failed",
            "error": str(exc),
            "run_id": str(globals().get("RUN_ID", "")),
        }
        print(f"[NB6] FAILED {adapter_key}: {exc}")
        continue

print('\n[NB6] SUMMARY')
print(json.dumps(NB6_RESULTS, indent=2, ensure_ascii=False))